# Pipeline de Entrenamiento en Español (Test A/B)

El objetivo de este notebook es replicar la arquitectura de entrenamiento (Baseline) implementada en el *Notebook 02*, pero utilizando el dataset traducido al español (`df_final_silver_es.parquet`). 

Esto nos permitirá ejecutar un **Test A/B de Idioma**: mediremos si el modelo sufre una degradación grave de F1-Macro al trabajar con texto traducido automáticamente en comparación con el texto nativo original.

In [1]:
import pandas as pd
import warnings

# Ocultamos advertencias estéticas de Pandas para mantener limpio el notebook
warnings.filterwarnings('ignore') 

# 1. CARGA DEL DATASET
# Apuntamos a la carpeta processed donde el Notebook 03 guardó el archivo final
ruta_datos = "../data/processed/df_final_silver_es.parquet"
df_es = pd.read_parquet(ruta_datos)

# 2. AUDITORÍA DE VOLUMEN
# shape[0] te da el número de filas, shape[1] el número de columnas
print(f"Dimensiones del dataset en español: {df_es.shape}")

# 3. AUDITORÍA DE ETIQUETAS PREDICTIVAS
# value_counts() cuenta cuántos tickets hay en cada categoría y los ordena de mayor a menor
print("\nDistribución de las colas de soporte (Objetivo Predictivo):")
print(df_es['queue'].value_counts())

Dimensiones del dataset en español: (23867, 7)

Distribución de las colas de soporte (Objetivo Predictivo):
queue
Technical Support                  7620
Product Support                    4807
Customer Service                   3907
IT Support                         3088
Billing and Payments               2627
Service Outages and Maintenance    1019
Sales and Pre-Sales                 799
Name: count, dtype: int64


### Paso 2: Limpieza Avanzada (spaCy - Motor Español)

Para maximizar la densidad de información antes de la vectorización matemática, aplicamos técnicas de Procesamiento de Lenguaje Natural (NLP). 

Utilizamos el motor `es_core_news_sm` de spaCy. El código comprobará de forma autónoma si este motor está instalado en el sistema y, de no ser así, procederá a su descarga automática. Posteriormente, el motor eliminará signos de puntuación, convertirá el texto a minúsculas, filtrará las *stop-words* (artículos, preposiciones) específicas del idioma español y extraerá el lema (la raíz de diccionario) de cada palabra.

In [2]:
import spacy
from spacy.cli import download
from tqdm.auto import tqdm

# 1. AUTOGESTIÓN DE DEPENDENCIAS
# Intentamos cargar el modelo español. Si salta un error (OSError), lo descargamos al vuelo.
try:
    nlp_es = spacy.load("es_core_news_sm")
except OSError:
    print("⚠️ Motor lingüístico español no detectado. Iniciando descarga automática...")
    download("es_core_news_sm")
    nlp_es = spacy.load("es_core_news_sm")
    print("✅ Descarga e instalación completadas.")

# 2. DEFINICIÓN DEL MOTOR DE LIMPIEZA
def clean_text_spacy_es(text):
    # Convertimos el texto bruto en un documento analizado gramaticalmente por la IA de spaCy
    doc = nlp_es(text)
    
    # Extraemos el lema (raíz) solo de las palabras que nos interesan
    clean_tokens = [
        token.lemma_.lower() # Lo pasamos todo a minúsculas y sacamos la raíz
        for token in doc 
        if token.is_alpha and not token.is_stop # is_alpha elimina números/signos, is_stop elimina palabras vacías
    ]
    
    # Volvemos a unir las raíces sueltas en una sola frase
    return " ".join(clean_tokens)

print("Iniciando limpieza lingüística con spaCy...")
print("Esto tomará alrededor de un minuto (tarea intensiva de CPU)...")

# 3. BARRA DE PROGRESO
# Registramos tqdm con Pandas para tener telemetría visual
tqdm.pandas(desc="Limpiando Texto")

# 4. EJECUCIÓN MASIVA
# El resultado lo guardamos en una columna nueva para mantener intacto el original
df_es['full_text_clean'] = df_es['full_text'].progress_apply(clean_text_spacy_es)

print("✅ Limpieza completada con éxito.")

# 5. AUDITORÍA
# Imprimimos el antes y el después de la fila 0 para comprobar que ha funcionado
print("\n--- AUDITORÍA DE LIMPIEZA VISUAL ---")
print(f"ORIGINAL: {df_es['full_text'].iloc[0]}")
print(f"LIMPIO:   {df_es['full_text_clean'].iloc[0]}")

Iniciando limpieza lingüística con spaCy...
Esto tomará alrededor de un minuto (tarea intensiva de CPU)...


Limpiando Texto: 100%|██████████| 23867/23867 [04:23<00:00, 90.65it/s] 

✅ Limpieza completada con éxito.

--- AUDITORÍA DE LIMPIEZA VISUAL ---
ORIGINAL: Interrupción de la cuenta Estimado equipo de atención al cliente,\n\nEstoy escribiendo para informar de un problema significativo con el portal de gestión de cuentas centralizada, que actualmente parece estar desconectado. Este corte está bloqueando el acceso a la configuración de la cuenta, lo que conduce a inconvenientes sustanciales. He intentado iniciar sesión varias veces utilizando diferentes navegadores y dispositivos, pero el problema persiste.\n\n¿Podría proporcionar una actualización sobre el estado de interrupción y un tiempo estimado para la resolución? Además, ¿hay alguna forma alternativa de acceder y administrar mi cuenta durante este tiempo de inactividad?
LIMPIO:   interrupción estimado equipo atención escribir informar problema significativo portal gestión cuenta centralizado actualmente desconectado corte bloquear acceso configuración conducir inconveniente sustancial intentar iniciar se

### Paso 3: División del Dataset (Train, Val, Test)

Para garantizar la validez del Test A/B, replicamos **exactamente la misma partición** que hicimos en el modelo inglés. 
Mapeamos nuestra variable de entrada `X` al texto limpio (`full_text_clean`) y nuestra variable objetivo `y` a las colas de soporte (`queue`). Dividiremos los datos en un 70% para Entrenamiento, 15% para Validación (ajuste de modelo) y 15% para Test Ciego.

> **Importante:** Utilizamos `stratify=y` para asegurar que las clases minoritarias (como *Service Outages*) mantengan la misma proporción exacta en los tres conjuntos, evitando sesgar el entrenamiento.

In [3]:
from sklearn.model_selection import train_test_split

# 1. DEFINICIÓN DE VARIABLES
X = df_es['full_text_clean'] # El texto purificado
y = df_es['queue']           # La etiqueta a predecir

print("Realizando la partición de datos (70% Train, 15% Val, 15% Test)...")

# 2. PRIMERA PARTICIÓN (Separamos el 70% de Train)
# Sacamos un 30% temporal (temp) que luego dividiremos por la mitad
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30,      # Dejamos un 30% fuera del entrenamiento
    stratify=y,          # Estratificación: Repartimos las averías equitativamente
    random_state=42      # Semilla fija: El inglés y el español se separarán exactamente por las mismas filas
)

# 3. SEGUNDA PARTICIÓN (Dividimos el 30% sobrante en dos mitades del 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,      # Mitad y mitad
    stratify=y_temp,     # Volvemos a estratificar el grupo pequeño
    random_state=42
)

# 4. AUDITORÍA DE DIMENSIONES
print("\n--- DIMENSIONES DE LOS CONJUNTOS ---")
print(f"X_train (Entrenamiento): {X_train.shape[0]} tickets")
print(f"X_val   (Validación)   : {X_val.shape[0]} tickets")
print(f"X_test  (Prueba Ciega) : {X_test.shape[0]} tickets")

# Comprobación de que la estratificación ha funcionado
print("\nProporción de 'Service Outages' en Train: {:.2f}%".format(
    (y_train == 'Service Outages and Maintenance').mean() * 100
))
print("Proporción de 'Service Outages' en Val:   {:.2f}%".format(
    (y_val == 'Service Outages and Maintenance').mean() * 100
))

Realizando la partición de datos (70% Train, 15% Val, 15% Test)...

--- DIMENSIONES DE LOS CONJUNTOS ---
X_train (Entrenamiento): 16706 tickets
X_val   (Validación)   : 3580 tickets
X_test  (Prueba Ciega) : 3581 tickets

Proporción de 'Service Outages' en Train: 4.27%
Proporción de 'Service Outages' en Val:   4.27%


### Paso 4: Vectorización TF-IDF (Matriz Matemática)

Los algoritmos de Machine Learning no entienden texto, solo matrices numéricas.
Utilizamos `TfidfVectorizer` para transformar las frases en vectores. Esta técnica no solo cuenta cuántas veces aparece una palabra, sino que pondera su importancia: da mucho peso estadístico a palabras raras que discriminan bien una avería ("latencia", "fibra"), y quita peso a palabras comunes ("hola", "gracias").

> **Regla de Oro de MLOps (Data Leakage):** El vectorizador **SOLO** debe aprender el vocabulario (`fit`) sobre el conjunto de Entrenamiento (`X_train`). Los conjuntos de Validación y Test deben transformarse a ciegas usando estrictamente el diccionario que aprendió en el Train.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. INSTANCIAMOS EL MOTOR MATEMÁTICO
# Usamos exactamente los mismos parámetros que en inglés para que la comparativa sea válida
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000, # Nos quedamos con las 10.000 palabras/pares más importantes
    ngram_range=(1, 2)  # Permite palabras sueltas ("router") y pares ("no funciona")
)

print("Entrenando el vocabulario (TF-IDF) solo con X_train...")

# 2. ENTRENAMIENTO Y TRANSFORMACIÓN (SOLO TRAIN)
# El método .fit_transform() hace dos cosas a la vez:
# A) Aprende el diccionario de palabras (fit)
# B) Transforma las frases en una matriz de números (transform)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

print("Transformando los conjuntos de validación a ciegas...")

# 3. TRANSFORMACIÓN PURA (VAL Y TEST)
# ¡Ojo! Solo usamos .transform(). Si usáramos .fit_transform() aquí, arruinaríamos el experimento.
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("✅ Vectorización completada.")

# 4. AUDITORÍA DE MATRICES
print("\n--- DIMENSIONES DE LAS MATRICES TF-IDF ---")
print(f"X_train_tfidf: {X_train_tfidf.shape}")
print(f"X_val_tfidf:   {X_val_tfidf.shape}")
print(f"X_test_tfidf:  {X_test_tfidf.shape}")

Entrenando el vocabulario (TF-IDF) solo con X_train...
Transformando los conjuntos de validación a ciegas...
✅ Vectorización completada.

--- DIMENSIONES DE LAS MATRICES TF-IDF ---
X_train_tfidf: (16706, 10000)
X_val_tfidf:   (3580, 10000)
X_test_tfidf:  (3581, 10000)


### Paso 5: Entrenamiento del Modelo Baseline (Multinomial Naive Bayes)

El objetivo de este paso es establecer la "marca de agua" en español. Utilizamos **Multinomial Naive Bayes (MNB)** igual que hicimos en inglés, sin alterar ningún parámetro. 

Al mantener el algoritmo y los hiperparámetros constantes, cualquier diferencia en el F1-Macro final será imputable **exclusivamente al cambio de idioma** (la variable aislada de nuestro Test A/B).

> **Métricas de MLOps:** Mantenemos la sonda del cronómetro alrededor del método `.fit()` para registrar de forma precisa el coste computacional del entrenamiento en segundos.

In [5]:
from sklearn.naive_bayes import MultinomialNB
import time

# 1. INSTANCIAMOS EL MODELO
# Parámetros por defecto para emular el experimento base
baseline_model_mnb = MultinomialNB()

# 2. CRONÓMETRO E INYECCIÓN DE DATOS (EL ENTRENAMIENTO)
print("Iniciando entrenamiento del modelo MNB en Español...")
start_train_time = time.time() # Sonda de inicio

# Magia algorítmica: El modelo cruza la matriz de 10.000 palabras con las etiquetas
baseline_model_mnb.fit(X_train_tfidf, y_train)

end_train_time = time.time() # Sonda de fin

# 3. CÁLCULO DE MLOPS
train_time_sec = round(end_train_time - start_train_time, 4)

print("--- Entrenamiento Completado ---")
print(f"Tiempo de entrenamiento (train_time_sec): {train_time_sec} segundos")

Iniciando entrenamiento del modelo MNB en Español...
--- Entrenamiento Completado ---
Tiempo de entrenamiento (train_time_sec): 0.0425 segundos


### Paso 6: Inferencia y El Veredicto (Test A/B)

En esta fase ejecutamos el modelo entrenado contra el conjunto ciego de Validación (`X_val_tfidf`). 

> **Métricas Operativas:** Volvemos a calcular la latencia de inferencia en milisegundos para asegurar que el despliegue en una futura API seguirá siendo en tiempo real.

> **Métricas de Rendimiento (El Veredicto):** Extraeremos el **F1-Macro**. Si este F1 ronda o supera el 0.3347 que obtuvimos en el Notebook 02, habremos validado matemáticamente que la traducción automática no ha destruido el contexto predictivo de los clientes y podremos continuar el TFM en español.

In [6]:
from sklearn.metrics import classification_report, f1_score
import time

# 1. CRONÓMETRO DE INFERENCIA
print("Iniciando predicción sobre el conjunto de Validación (X_val) en Español...")
start_inf_time = time.time()

# El modelo se examina usando la matriz TF-IDF de validación
y_pred = baseline_model_mnb.predict(X_val_tfidf)

end_inf_time = time.time()

# 2. LATENCIA OPERATIVA
# (Milisegundos)
inference_time_ms = round((end_inf_time - start_inf_time) * 1000, 2)
print(f"Latencia de inferencia: {inference_time_ms} ms\n")

# 3. REPORTE DE NEGOCIO
# Incluimos zero_division=0 para evitar los avisos rojos si falla estrepitosamente en alguna clase minoritaria
print("--- REPORTE DE CLASIFICACIÓN (ESPAÑOL) ---")
print(classification_report(y_val, y_pred, zero_division=0))

# 4. EXTRACCIÓN DE KPIS CLAVE
f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)

report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)

print("-" * 50)
print(f"Veredicto Inglés (Notebook 02) -> F1-Macro: 0.3347 | Averías: 0.2775")
print(f"Veredicto Español (Actual)     -> F1-Macro: {f1_macro} | Averías: {f1_minority}")
print("-" * 50)

Iniciando predicción sobre el conjunto de Validación (X_val) en Español...
Latencia de inferencia: 2.42 ms

--- REPORTE DE CLASIFICACIÓN (ESPAÑOL) ---
                                 precision    recall  f1-score   support

           Billing and Payments       0.93      0.62      0.74       394
               Customer Service       0.31      0.45      0.37       586
                     IT Support       0.76      0.05      0.09       463
                Product Support       0.44      0.25      0.32       721
            Sales and Pre-Sales       0.00      0.00      0.00       120
Service Outages and Maintenance       0.88      0.05      0.09       153
              Technical Support       0.44      0.78      0.56      1143

                       accuracy                           0.45      3580
                      macro avg       0.54      0.31      0.31      3580
                   weighted avg       0.52      0.45      0.40      3580

-------------------------------------------

### Paso 7: Registro del Experimento (Tracker en Español)

Para mantener la trazabilidad de nuestra estrategia *Dual-Track*, inicializamos un registro de experimentos específico para el idioma español (`tracker_df_es`). 

En esta celda documentamos el rendimiento de nuestro modelo base (Multinomial Naive Bayes sin balanceo). Este bajo rendimiento (especialmente en la clase minoritaria) queda registrado como justificación de negocio para implementar algoritmos más complejos en las siguientes iteraciones.

In [7]:
import pandas as pd

# 1. CREACIÓN DEL TRACKER (Se ejecuta una sola vez para inicializar la tabla)
columnas_tracker = ['exp_id', 'target_level', 'vectorization', 'model', 'balancing', 
                    'train_time_sec', 'inference_time_ms', 'f1_macro', 'f1_minority_class']

tracker_df_es = pd.DataFrame(columns=columnas_tracker)

# 2. EMPAQUETADO DE RESULTADOS
# Usamos el prefijo 'ES_' para diferenciarlo claramente de los experimentos en inglés
nuevo_experimento = {
    'exp_id': 'ES_L1_TFIDF_MNB_NONE',
    'target_level': 'queue',
    'vectorization': 'tfidf',
    'model': 'mnb',
    'balancing': 'none',
    'train_time_sec': train_time_sec,          # Calculado en el Paso 5
    'inference_time_ms': inference_time_ms,    # Calculado en el Paso 6
    'f1_macro': f1_macro,                      # Calculado en el Paso 6
    'f1_minority_class': f1_minority           # Calculado en el Paso 6
}

# 3. INYECCIÓN Y GUARDADO
# Concatenamos la nueva fila al DataFrame vacío
tracker_df_es = pd.concat([tracker_df_es, pd.DataFrame([nuevo_experimento])], ignore_index=True)

# 4. AUDITORÍA VISUAL
print("--- REGISTRO DE EXPERIMENTOS ESPAÑOL (TRACKER) ---")
display(tracker_df_es)

--- REGISTRO DE EXPERIMENTOS ESPAÑOL (TRACKER) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class
0,ES_L1_TFIDF_MNB_NONE,queue,tfidf,mnb,none,0.0425,2.42,0.3088,0.087


### Paso Final: Serialización MLOps (Guardado a Disco - Español)

Siguiendo la arquitectura de Experimentación Desacoplada, guardamos las matrices TF-IDF y las etiquetas generadas en este pipeline de preparación en español. 

Al igual que en el idioma inglés, utilizamos el formato hipercomprimido `.npz` de SciPy para los tensores matemáticos y archivos `.csv` estándar para las etiquetas objetivo y el Tracker. A partir de ahora, todo entrenamiento en español se ejecutará en notebooks externos cargando estas piezas.

In [8]:
import os
import scipy.sparse
import pandas as pd

print("Iniciando volcado de matrices matemáticas a disco duro (Español)...")

# 1. DIRECTORIO DE FEATURES
features_dir = "../data/features"
os.makedirs(features_dir, exist_ok=True)

# 2. SERIALIZACIÓN DE MATRICES (X)
# OJO: Cambiamos el prefijo a 'es_' para no sobreescribir las matrices del inglés
scipy.sparse.save_npz(f"{features_dir}/es_X_train_tfidf.npz", X_train_tfidf)
scipy.sparse.save_npz(f"{features_dir}/es_X_val_tfidf.npz", X_val_tfidf)
scipy.sparse.save_npz(f"{features_dir}/es_X_test_tfidf.npz", X_test_tfidf)
print("✅ Matrices X (TF-IDF) en español guardadas.")

# 3. SERIALIZACIÓN DE ETIQUETAS (y)
pd.DataFrame(y_train).to_csv(f"{features_dir}/es_y_train.csv", index=False)
pd.DataFrame(y_val).to_csv(f"{features_dir}/es_y_val.csv", index=False)
pd.DataFrame(y_test).to_csv(f"{features_dir}/es_y_test.csv", index=False)
print("✅ Etiquetas y (Target) en español guardadas.")

# 4. ACTUALIZACIÓN Y GUARDADO DEL TRACKER ESPAÑOL
# Inyectamos la nueva columna para igualar la estructura al roadmap dual
if 'hyperparameters' not in tracker_df_es.columns:
    tracker_df_es['hyperparameters'] = 'baseline default'

# Guardamos el tracker como un archivo independiente
tracker_df_es.to_csv("../data/processed/tracker_es.csv", index=False)
print("✅ Tracker de experimentos maestro (Español) guardado en disco.")

print("\n🚀 PROCESO MLOPS ESPAÑOL COMPLETADO. Notebook 04 clausurado.")

Iniciando volcado de matrices matemáticas a disco duro (Español)...
✅ Matrices X (TF-IDF) en español guardadas.
✅ Etiquetas y (Target) en español guardadas.
✅ Tracker de experimentos maestro (Español) guardado en disco.

🚀 PROCESO MLOPS ESPAÑOL COMPLETADO. Notebook 04 clausurado.
